# 🟠 Galileo — Tier 1: Error path

Same scenario as the other two platforms. **This is the hero scenario for Galileo's Tool Error Rate auto-metric.**

When the agent asks `get_ads_id("Nobody McNobody")` and the tool returns an `ERROR:` string, Galileo's automatic Tool Error Rate metric (if it keys on payload patterns) should surface this trace immediately — no eval configuration needed.

**What to look for in the Galileo UI**:
- Project from `GALILEO_PROJECT` → log stream `default` → filter tag `tier1` + `error_path`
- Click the trace → **Insights** panel
- Tool Error Rate / Action Completion metrics should reflect the failed tool
- Compare with the Langfuse / LangSmith error_path traces — same data, but the Insights panel is what makes Galileo's view different

In [ ]:
import os, sys, pathlib
from dotenv import load_dotenv

ROOT = pathlib.Path().resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

os.environ.setdefault("GALILEO_PROJECT", "observability-comparison")
os.environ.setdefault("GALILEO_LOG_STREAM", "default")

from galileo.handlers.langchain import GalileoCallback
from langchain_core.messages import HumanMessage
from shared.workflow import build_agent, SCENARIOS

scenario = next(s for s in SCENARIOS if s["id"] == "error_path")
print(f"Scenario: {scenario['id']}")
print(f"Prompt  : {scenario['prompt']}")
print(f"Galileo project: {os.environ['GALILEO_PROJECT']}, log stream: {os.environ['GALILEO_LOG_STREAM']}")

handler = GalileoCallback()

In [ ]:
agent = build_agent(prompt_source="galileo")

config = {
    "callbacks": [handler],
    "run_name": scenario["id"],
    "tags": ["tier1", "error_path", "galileo"],
    "metadata": {
        "scenario_id": scenario["id"],
        "session_id": "tier1-error-path-demo",
    },
}

out = agent.invoke({"messages": [HumanMessage(content=scenario["prompt"])]}, config=config)

for i, m in enumerate(out["messages"]):
    role = type(m).__name__
    content = (m.content if isinstance(m.content, str) else str(m.content))[:200]
    tool_calls = getattr(m, "tool_calls", None) or []
    print(f"  [{i}] {role:<14} {content}")
    for tc in tool_calls:
        name = tc.get("name") if isinstance(tc, dict) else tc.name
        print(f"        tool_call -> {name}")

# Flush — best-effort across SDK versions
for attr in ("flush", "_flush", "close"):
    fn = getattr(handler, attr, None)
    if callable(fn):
        try:
            fn()
            break
        except Exception:
            pass
import time; time.sleep(3)
print("\nDone. Open https://app.galileo.ai -> Insights panel will show metric scores.")

## Deck takeaway

**Galileo on errors**: the trace lands with the same data as Langfuse/LangSmith, but the **Insights panel auto-populates** with metric scores. Look specifically at:
- **Tool Error Rate** — whether the platform considers a tool returning `ERROR:` as a tool error (may vary by SDK version)
- **Action Completion** — should reflect that the multi-step task didn't complete its goal
- **Context Adherence** — should be high (agent didn't fabricate; it acknowledged the error)

This is the deck slide pair: same data in 3 UIs, only Galileo's UI is opinionated about what "good" looks like out of the box.